# Train Model

## Import thư viện

In [2]:
import os
import re
import math
import argparse
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Optional, List
import copy
import json

import yaml
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

## 1. Settings

In [ ]:
config = {
	"defaults": {
		"task": "topic",
		"profile": "default",
		"seed": 42,
	},
	"common": {
		"model": {
			"name": "vinai/phobert-base",
			"max_length": 128,
		},
		"data": {
			"use_context_prev": True,
			"use_context_next": True,
		},
		"training": {
			"epochs": 5,
			"train_batch_size": 16,
			"eval_batch_size": 32,
			"learning_rate": 2.0e-5,
			"weight_decay": 0.01,
			'warmup_steps': 0.1,
			"use_class_weights": True,
			"weight_method": "sqrt_inverse",
			"gradient_accumulation_steps": 1,
			"max_grad_norm": 1.0,
			"label_smoothing_factor": 0.0,
			"fp16": "auto",
			"metric_for_best_model": "macro_f1",
		},
		"early_stopping": {
			"enabled": True,
			"patience": 2,
			"threshold": 0.0,
		},
		"neuro_symbolic": {
			"enabled": True,
			"constraint_lambda": 0.3,
			"exactly_one_weight": 1.0,
			"implication_weight": 0.5,
			"negation_weight": 0.5,
			"inference_alpha": 0.3,
			"min_confidence": 0.3,
		},
        "experiment": {
			"alpha_grid": [0.3, 0.5, 0.7, 0.9],
		},
	},
	"tasks": {
		"topic": {
			"labels": ["E", "S_labor", "S_community", "S_product", "G", "Non_ESG"],
			"paths": {
				"train_data": "/data/labels/topic/train.parquet",
				"val_data": "/data/labels/topic/val.parquet",
				"test_data": "/data/labels/topic/test.parquet",
				"output_dir": "/output/models/topic_classifier",
			},
			"profiles": {
				"default": {},
			},
		},
		"action": {
			"labels": ["Implemented", "Planning", "Indeterminate"],
			"paths": {
				"train_data": "/data/labels/action/train.parquet",
				"val_data": "/data/labels/action/val.parquet",
				"test_data": "/data/labels/action/test.parquet",
				"output_dir": "/output/models/action_classifier",
			},
			"profiles": {
				"default": {},
			},
		},
	},
}

## 2. Grounded Rules

In [4]:
@dataclass
class LabelingRule:
    name: str
    patterns: List[str]
    source: str
    description: str
    weight: float = 1.0

# --- ENVIRONMENT (GRI 300 Series) ---
GRI_ENVIRONMENT_RULES = [
    LabelingRule(
        name="GRI301_Materials",
        patterns=[
            r"\b(nguyên vật liệu|vật liệu tái chế|tái chế|recycled material)",
            r"\b(tiêu thụ nguyên liệu|material consumption)\b",
        ],
        source="GRI 301: Materials 2016",
        description="Materials usage and recycling",
    ),
    LabelingRule(
        name="GRI302_Energy",
        patterns=[
            r"\b(năng lượng|energy|kWh|MWh|GWh|MW)\b",
            r"\b(tiêu thụ năng lượng|energy consumption|cường độ năng lượng)\b",
            r"\b(năng lượng tái tạo|renewable energy|điện mặt trời|solar|wind)\b",
            r"\b(tiết kiệm năng lượng|energy saving|energy efficiency)\b",
        ],
        source="GRI 302: Energy 2016",
        description="Energy consumption, intensity, and reduction",
    ),
    LabelingRule(
        name="GRI303_Water",
        patterns=[
            r"\b(nước|water|m3|tài nguyên nước|nguồn nước)\b",
            r"\b(nước thải|wastewater|xử lý nước|water treatment)\b",
        ],
        source="GRI 303: Water and Effluents 2018",
        description="Water withdrawal, consumption, and discharge",
    ),
    LabelingRule(
        name="GRI305_Emissions",
        patterns=[
            r"\b(chuyển đổi xanh|green transition|chuyển dịch năng lượng|energy transition)\b",
            r"\b(kiểm kê khí nhà kính|ghg inventory|phát thải ròng bằng không|tín chỉ carbon|carbon credit)\b",
            r"\b(khí thải|phát thải|emission|CO2|carbon|GHG)\b",
            r"\b(Scope\s*[123]|phạm vi\s*[123])\b",
            r"\b(carbon footprint|dấu chân carbon|khí nhà kính)\b",
            r"\b(net[\-\s]?zero|trung hòa carbon|carbon neutral)\b",
            r"\b(giảm phát thải|emission reduction|carbon offset)\b",
        ],
        source="GRI 305: Emissions 2016",
        description="GHG emissions, reduction, intensity",
    ),
    LabelingRule(
        name="GRI306_Waste",
        patterns=[
            r"\b(kinh tế tuần hoàn|circular economy|rác thải nhựa|plastic waste|vật liệu phân hủy sinh học)\b",
            r"\b(chất thải|rác thải|waste|hazardous waste)\b",
            r"\b(xử lý chất thải|waste management|waste disposal)\b",
            r"\b(ô nhiễm|pollution|contamination)\b",
        ],
        source="GRI 306: Waste 2020",
        description="Waste generation, disposal, recycling",
    ),
    LabelingRule(
        name="GRI304_Biodiversity",
        patterns=[
            r"\b(đa dạng sinh học|biodiversity|hệ sinh thái|ecosystem)\b",
            r"\b(bảo tồn|conservation|rừng|forest|deforestation)\b",
        ],
        source="GRI 304: Biodiversity 2016",
        description="Biodiversity impacts and conservation",
    ),
    LabelingRule(
        name="Climate_Finance",
        patterns=[
            r"\b(khung tài chính xanh|green framework|esg loan|khoản vay xanh|tài trợ dự án xanh)\b",
            r"\b(tín dụng xanh|green credit|trái phiếu xanh|green bond)\b",
            r"\b(tài chính xanh|green finance|tài chính bền vững)\b",
            r"\b(tài chính khí hậu|climate finance)\b",
        ],
        source="GRI 201 + TCFD Guidelines",
        description="Green financial instruments and climate finance",
    ),
]

# --- SOCIAL: LABOR (GRI 400 Series, 401-406) ---
GRI_SOCIAL_LABOR_RULES = [
    LabelingRule(
        name="GRI401_Employment",
        patterns=[
            r"\b(tuyển dụng|recruitment|nhân sự mới|new hires)\b",
            r"\b(phúc lợi|benefits|chế độ đãi ngộ|compensation)\b",
            r"\b(lương|salary|thu nhập|income|thưởng|bonus)\b",
        ],
        source="GRI 401: Employment 2016",
        description="Employment practices, benefits, retention",
    ),
    LabelingRule(
        name="GRI403_OHS",
        patterns=[
            r"\b(an toàn lao động|occupational health|safety|ATLĐ)\b",
            r"\b(sức khỏe nghề nghiệp|workplace safety)\b",
            r"\b(tai nạn lao động|work injury|incident)\b",
        ],
        source="GRI 403: Occupational Health and Safety 2018",
        description="Workplace health and safety",
    ),
    LabelingRule(
        name="GRI404_Training",
        patterns=[
            r"\b(giữ chân nhân tài|talent retention|đánh giá hiệu quả công việc|performance review|lộ trình thăng tiến)\b",
            r"\b(đào tạo|training|phát triển nhân sự|staff development)\b",
            r"\b(nâng cao năng lực|capacity building|skill)\b",
            r"\b(giờ đào tạo|training hours|chương trình đào tạo)\b",
        ],
        source="GRI 404: Training and Education 2016",
        description="Employee training and development",
    ),
    LabelingRule(
        name="GRI405_Diversity",
        patterns=[
            r"\b(bình đẳng giới|gender equality|đa dạng|diversity)\b",
            r"\b(nữ giới|female|phụ nữ|women in leadership)\b",
            r"\b(hòa nhập|inclusion|công bằng|equity)\b",
        ],
        source="GRI 405: Diversity and Equal Opportunity 2016",
        description="Diversity, equity, and inclusion",
    ),
    LabelingRule(
        name="GRI402_LaborRelations",
        patterns=[
            r"\b(người lao động|CBNV|cán bộ nhân viên|employee)\b",
            r"\b(quan hệ lao động|labor relations|công đoàn|union)\b",
            r"\b(môi trường làm việc|work environment|workplace)\b",
            r"\b(văn hóa doanh nghiệp|corporate culture)\b",
        ],
        source="GRI 402: Labor/Management Relations 2016",
        description="Labor relations and work conditions",
    ),
]

# --- SOCIAL: COMMUNITY (GRI 413) ---
GRI_SOCIAL_COMMUNITY_RULES = [
    LabelingRule(
        name="GRI413_Community",
        patterns=[
            r"\b(cộng đồng|community|địa phương|local)\b",
            r"\b(từ thiện|charity|thiện nguyện|volunteer)\b",
            r"\b(trách nhiệm xã hội|social responsibility|CSR)\b",
            r"\b(an sinh xã hội|social welfare)\b",
            r"\b(học bổng|scholarship|quỹ xã hội|social fund)\b",
            r"\b(phát triển cộng đồng|community development)\b",
            r"\b(cứu trợ|relief|hiến máu|blood donation|tài trợ giáo dục|tài trợ y tế)\b"
        ],
        source="GRI 413: Local Communities 2016",
        description="Community engagement and social programs",
    ),
]

# --- SOCIAL: PRODUCT RESPONSIBILITY (GRI 416-418) ---
GRI_SOCIAL_PRODUCT_RULES = [
    LabelingRule(
        name="GRI416_CustomerHealth",
        patterns=[
            r"\b(bảo vệ.*khách hàng|consumer protection)\b",
            r"\b(quyền lợi khách hàng|customer rights)\b",
        ],
        source="GRI 416: Customer Health and Safety 2016",
        description="Customer health and safety",
    ),
    LabelingRule(
        name="GRI417_Marketing",
        patterns=[
            r"\b(minh bạch thông tin sản phẩm|product labeling)\b",
            r"\b(chất lượng dịch vụ|service quality)\b",
            r"\b(trải nghiệm khách hàng|customer experience)\b",
        ],
        source="GRI 417: Marketing and Labeling 2016",
        description="Product/service marketing and labeling",
    ),
    LabelingRule(
        name="GRI418_Privacy",
        patterns=[
            r"\b(bảo mật|bảo vệ dữ liệu|data protection|privacy)\b",
            r"\b(an toàn thông tin|information security|an ninh mạng|cybersecurity)\b",
            r"\b(dữ liệu cá nhân|personal data|pdpa|nddp|nghị định 13|chuẩn pci dss)\b"
        ],
        source="GRI 418: Customer Privacy 2016",
        description="Customer data privacy and security",
    ),
    LabelingRule(
        name="FinancialInclusion",
        patterns=[
            r"\b(tài chính toàn diện|financial inclusion)\b",
            r"\b(giáo dục tài chính|financial literacy)\b",
            r"\b(tiếp cận tài chính|access to finance)\b",
        ],
        source="SDG 8: Decent Work, SDG 10: Reduced Inequality",
        description="Financial inclusion and literacy",
    ),
]

# --- GOVERNANCE (GRI 200 Series + GRI 2 General) ---
GRI_GOVERNANCE_RULES = [
    LabelingRule(
        name="GRI205_AntiCorruption",
        patterns=[
            r"\b(chống tham nhũng|anti-corruption|liêm chính|integrity)\b",
            r"\b(đạo đức kinh doanh|business ethics|code of conduct)\b",
            r"\b(phòng chống rửa tiền|aml|biết khách hàng là ai|kyc|chống tài trợ khủng bố|cft)\b",
            r"\b(bảo vệ người tố giác|whistleblower|xung đột lợi ích|conflict of interest)\b"
        ],
        source="GRI 205: Anti-corruption 2016",
        description="Anti-corruption policies and practices",
    ),
    LabelingRule(
        name="GRI2_Governance",
        patterns=[
            r"\b(quản trị công ty|corporate governance)\b",
            r"\b(hội đồng quản trị|board of directors|HĐQT)\b",
            r"\b(minh bạch|transparency|công bố thông tin|disclosure)\b",
            r"\b(ủy ban esg|esg committee|ban chỉ đạo esg|tích hợp esg|esg integration)\b"
        ],
        source="GRI 2: General Disclosures 2021",
        description="Corporate governance structure",
    ),
    LabelingRule(
        name="RiskManagement",
        patterns=[
            r"\b(quản trị rủi ro|risk management|quản lý rủi ro)\b",
            r"\b(kiểm soát nội bộ|internal control|kiểm toán nội bộ)\b",
            r"\b(tuân thủ|compliance|quy định|regulation)\b",
        ],
        source="GRI 2 + Basel III Framework",
        description="Risk management and compliance",
    ),
    LabelingRule(
        name="Audit_Oversight",
        patterns=[
            r"\b(kiểm toán|audit|giám sát|oversight|supervision)\b",
            r"\b(ban kiểm soát|supervisory board)\b",
        ],
        source="GRI 2: General Disclosures 2021",
        description="Audit and supervisory functions",
    ),
]

# ACTIONABILITY CLASSIFICATION RULES

# --- IMPLEMENTED: Based on Bloom's Taxonomy Level 3-6 (Apply, Analyze, Evaluate, Create) ---
IMPLEMENTED_VERBS = LabelingRule(
    name="Bloom_HighLevel_Verbs",
    patterns=[
        # Level 3-6 past tense verbs indicating completed actions
        r"\b(đã triển khai|đã thực hiện|đã hoàn thành|đã đạt được)\b",
        r"\b(đã giảm|đã tăng|đã tiết kiệm|đã cắt giảm|đã xử lý)\b",
        r"\b(hoàn thành|ghi nhận|đạt được|thực hiện được)\b",
        r"\b(triển khai thành công|vận hành|ứng dụng|áp dụng)\b",
    ],
    source="Anderson & Krathwohl (2001). Bloom's Taxonomy Revised [6]",
    description="High-level cognitive verbs in past tense indicating completed actions",
)

IMPLEMENTED_EVIDENCE = LabelingRule(
    name="Quantitative_Results",
    patterns=[
        # Numbers with ESG-relevant units (quantified outcomes)
        r"(?:đã|năm \d{4}).{0,50}?\d+\s*(?:%|tỷ|triệu|nghìn|tấn|kWh|MWh|CO2|giờ)",
        r"\d+\s*(%|tỷ|triệu|nghìn|tấn|kWh|MWh).*?(so với|giảm|tăng|đạt)",
        # Temporal anchors in the past
        r"trong năm (2019|2020|2021|2022|2023)\b",
        r"năm (2019|2020|2021|2022|2023)\b.*?(đạt|hoàn thành|thực hiện)",
    ],
    source="Florstedt, Fahlbusch & Sontheimer (2025) [4] + GRI 'Quantification Principle'",
    description="Quantitative evidence of past performance",
)

# --- PLANNING: Based on future-oriented commitment language ---
PLANNING_INDICATORS = LabelingRule(
    name="Future_Commitment",
    patterns=[
        r"\b(sẽ|dự kiến|kế hoạch|định hướng|mục tiêu)\b",
        r"\b(hướng tới|phấn đấu|đặt mục tiêu|cam kết.*sẽ)\b",
        r"\b(triển khai trong|thực hiện trong giai đoạn)\b",
        # Future time markers
        r"(đến năm|vào năm|mục tiêu.*?năm)\s*(2025|2026|2027|2028|2029|2030|2050)\b",
        r"\b(net zero|trung hòa carbon).*?(2030|2040|2050)\b",
        r"(lộ trình|roadmap).*?(2025|2030|2050)",
    ],
    source="Florstedt, Fahlbusch & Sontheimer (2025) [4]",
    description="Forward-looking statements with specific targets",
)

# --- INDETERMINATE: Hedging and Boosting indicators ---

HEDGING_INDICATORS = LabelingRule(
    name="Hedging_Vagueness",
    patterns=[
        # Hedges (certainty reducers) — adapted from Hyland (2005)
        r"\b(có thể|perhaps|maybe|might)\b",
        r"\b(phần nào|somewhat|to some extent)\b",
        r"\b(tương đối|relatively|fairly)\b",
        # Vietnamese hedging expressions
        r"\b(ngày càng|không ngừng|liên tục|dần dần)\b",
        r"\b(góp phần|đóng góp vào|hỗ trợ)\b",
    ],
    source="Hyland (2005). Metadiscourse [3]; Crismore et al. (1993) [5]",
    description="Hedging markers that reduce commitment certainty",
)

BOOSTING_INDICATORS = LabelingRule(
    name="Boosting_Exaggeration",
    patterns=[
        # Boosters (certainty amplifiers without evidence) — adapted from Hyland (2005)
        r"\b(luôn luôn|always|definitely|certainly)\b",
        r"\b(rất|highly|extremely|hoàn toàn|absolutely)\b",
        # Vietnamese boosting/marketing language
        r"\b(hàng đầu|tiên phong|dẫn đầu|leading|pioneer)\b",
        r"\b(xuất sắc|outstanding|vượt trội|world-class)\b",
    ],
    source="Hyland (2005). Metadiscourse [3]",
    description="Boosting markers that amplify claims without evidence",
)

VAGUE_COMMITMENT = LabelingRule(
    name="Vague_Commitment_Language",
    patterns=[
        # Commitment without specifics — from Florstedt, Fahlbusch & Sontheimer (2025)
        r"\b(cam kết|hướng tới|tăng cường|đẩy mạnh|tiếp tục)\b",
        r"\b(chú trọng|quan tâm|ưu tiên|nỗ lực)\b",
        r"\b(phát triển bền vững|trách nhiệm xã hội)\b",
        r"\b(nâng cao nhận thức|nâng cao|cải thiện)\b",
        r"\b(đang nghiên cứu|xem xét|trong quá trình|từng bước)\b",
        r"\b(chung tay|góp sức|kêu gọi|đồng hành cùng)\b"
    ],
    source="Florstedt, Fahlbusch & Sontheimer (2025) [4] — 'cheap talk' indicators",
    description="Vague commitment language without actionable specifics",
)

ALL_TOPIC_RULES = {
    "E": GRI_ENVIRONMENT_RULES,
    "S_labor": GRI_SOCIAL_LABOR_RULES,
    "S_community": GRI_SOCIAL_COMMUNITY_RULES,
    "S_product": GRI_SOCIAL_PRODUCT_RULES,
    "G": GRI_GOVERNANCE_RULES,
}

ALL_ACTION_RULES = {
    "Implemented": [IMPLEMENTED_VERBS, IMPLEMENTED_EVIDENCE],
    "Planning": [PLANNING_INDICATORS],
    "Indeterminate": [HEDGING_INDICATORS, BOOSTING_INDICATORS, VAGUE_COMMITMENT],
}

def match_topic_grounded(text: str, context: str = "", section: str = "") -> tuple[str, float, list[str]]:
    text_lower = text.lower()
    ctx_lower = (f"{context} {text}").lower() if context else text_lower
    section_lower = section.lower() if section else ""
    
    scores = {t: 0.0 for t in ALL_TOPIC_RULES}
    matched = {t: [] for t in ALL_TOPIC_RULES}
    
    for topic, rules in ALL_TOPIC_RULES.items():
        for rule in rules:
            for pattern in rule.patterns:
                if re.search(pattern, text_lower, re.IGNORECASE):
                    scores[topic] += 0.4 * rule.weight
                    matched[topic].append(rule.name)
                    break  # Count each rule once
                elif re.search(pattern, ctx_lower, re.IGNORECASE):
                    scores[topic] += 0.1 * rule.weight
                    matched[topic].append(f"{rule.name}(ctx)")
                    break
    
    best_topic = max(scores, key=scores.get)
    best_score = scores[best_topic]
    
    if best_score < 0.3:
        return "Non_ESG", 0.5, []
    
    return best_topic, min(best_score, 1.0), matched[best_topic]


def match_actionability_grounded(text: str, context: str = "") -> tuple[str, float, list[str]]:
    """
    Match actionability using grounded rules (Bloom's Taxonomy + Hedging/Boosting).
    
    Returns:
        (label, confidence, matched_rules) — e.g., ("Implemented", 0.7, ["Bloom_HighLevel_Verbs"])
    """
    text_lower = text.lower()
    ctx_lower = (f"{context} {text}").lower() if context else text_lower
    
    scores = {label: 0.0 for label in ALL_ACTION_RULES}
    matched = {label: [] for label in ALL_ACTION_RULES}
    
    for label, rules in ALL_ACTION_RULES.items():
        for rule in rules:
            for pattern in rule.patterns:
                if re.search(pattern, text_lower, re.IGNORECASE):
                    scores[label] += 0.5 * rule.weight
                    matched[label].append(rule.name)
                    break
                elif re.search(pattern, ctx_lower, re.IGNORECASE):
                    scores[label] += 0.2 * rule.weight
                    matched[label].append(f"{rule.name}(ctx)")
                    break
    
    # Penalty: If Indeterminate but has numbers → likely not Indeterminate
    has_numbers = bool(re.search(r"\d+\s*(%|tỷ|triệu|nghìn|tấn|kg|kWh|MWh)", text_lower))
    has_future_year = bool(re.search(r"(2025|2026|2027|2028|2029|2030|2050)", text_lower))
    
    if has_numbers:
        scores["Indeterminate"] -= 0.3
    if has_future_year:
        scores["Indeterminate"] -= 0.2
        scores["Planning"] += 0.2
    
    best_label = max(scores, key=scores.get)
    best_score = scores[best_label]
    
    if best_score < 0.4:
        return "Indeterminate", 0.3, []
    
    return best_label, min(best_score, 1.0), matched[best_label]


def get_rule_provenance() -> dict:
    """
    Return a summary of all rules with their academic sources.
    Useful for thesis methodology section.
    """
    provenance = {}
    
    for topic, rules in ALL_TOPIC_RULES.items():
        provenance[topic] = [
            {"name": r.name, "source": r.source, "description": r.description, "num_patterns": len(r.patterns)}
            for r in rules
        ]
    
    for label, rules in ALL_ACTION_RULES.items():
        provenance[f"Action_{label}"] = [
            {"name": r.name, "source": r.source, "description": r.description, "num_patterns": len(r.patterns)}
            for r in rules
        ]
    
    return provenance


prov = get_rule_provenance()
print("=" * 60)
print("GROUNDED LABELING RULES SUMMARY")
print("=" * 60)

for category, rules in prov.items():
    print(f"\n--- {category} ---")
    for r in rules:
        print(f"  {r['name']:30s}  ({r['num_patterns']} patterns)  Source: {r['source']}")

# Test examples
print("\n\n" + "=" * 60)
print("TEST EXAMPLES")
print("=" * 60)

tests = [
    "Ngân hàng đã giảm phát thải CO2 được 15% so với năm 2022.",
    "Chúng tôi cam kết hướng tới phát triển bền vững.",
    "Mục tiêu đạt net zero vào năm 2050 theo lộ trình đã đề ra.",
    "Ngân hàng luôn quan tâm, chú trọng đến môi trường làm việc cho CBNV.",
    "Đã triển khai chương trình đào tạo cho 5.000 nhân viên trong năm 2023.",
]

for t in tests:
    topic, t_conf, t_rules = match_topic_grounded(t)
    action, a_conf, a_rules = match_actionability_grounded(t)
    print(f"\n\"{t}...\"")
    print(f"  Topic: {topic} (conf={t_conf:.2f}) — {t_rules}")
    print(f"  Action: {action} (conf={a_conf:.2f}) — {a_rules}")


GROUNDED LABELING RULES SUMMARY

--- E ---
  GRI301_Materials                (2 patterns)  Source: GRI 301: Materials 2016
  GRI302_Energy                   (4 patterns)  Source: GRI 302: Energy 2016
  GRI303_Water                    (2 patterns)  Source: GRI 303: Water and Effluents 2018
  GRI305_Emissions                (7 patterns)  Source: GRI 305: Emissions 2016
  GRI306_Waste                    (4 patterns)  Source: GRI 306: Waste 2020
  GRI304_Biodiversity             (2 patterns)  Source: GRI 304: Biodiversity 2016
  Climate_Finance                 (4 patterns)  Source: GRI 201 + TCFD Guidelines

--- S_labor ---
  GRI401_Employment               (3 patterns)  Source: GRI 401: Employment 2016
  GRI403_OHS                      (3 patterns)  Source: GRI 403: Occupational Health and Safety 2018
  GRI404_Training                 (4 patterns)  Source: GRI 404: Training and Education 2016
  GRI405_Diversity                (3 patterns)  Source: GRI 405: Diversity and Equal Opportunity 

## 3. Neuro Symbolic Settings

In [5]:
@dataclass
class Predicate:
    name: str
    patterns: list[str]
    source: str = ""
    
    def evaluate(self, text: str) -> bool:
        text_lower = text.lower()
        for pattern in self.patterns:
            if re.search(pattern, text_lower, re.IGNORECASE):
                return True
        return False
    
    def fuzzy_evaluate(self, text: str) -> float:
        text_lower = text.lower()
        match_count = 0
        total_patterns = len(self.patterns)
        for pattern in self.patterns:
            if re.search(pattern, text_lower, re.IGNORECASE):
                match_count += 1
        if total_patterns == 0:
            return 0.0
        return min(match_count / max(total_patterns, 1), 1.0)


@dataclass
class Constraint:
    name: str
    constraint_type: str  # "implication", "negation", "exactly_one", "mutual_exclusion"
    predicate: Optional[Predicate] = None
    target_label_idx: Optional[int] = None  # Index into label list
    source: str = ""
    
    def __post_init__(self):
        if self.constraint_type not in ("implication", "negation", "exactly_one", "mutual_exclusion"):
            raise ValueError(f"Invalid constraint type: {self.constraint_type}")


class PropositionalKnowledgeBase:
    def __init__(self, labels: list[str], task: str = "topic"):
        self.labels = labels
        self.task = task
        self.label_to_idx = {label: i for i, label in enumerate(labels)}
        self.constraints: list[Constraint] = []
        self.predicates: dict[str, list[Predicate]] = {}  # label -> predicates
        
        self._build_knowledge_base()
    
    def _build_knowledge_base(self):
        self.constraints.append(Constraint(
            name="exactly_one",
            constraint_type="exactly_one",
            source="Classification axiom: single-label constraint",
        ))
        
        rule_source = ALL_TOPIC_RULES if self.task == "topic" else ALL_ACTION_RULES
        
        for label, rules in rule_source.items():
            if label not in self.label_to_idx:
                continue
            
            label_idx = self.label_to_idx[label]
            label_predicates = []
            
            for rule in rules:
                # Each LabelingRule -> a Predicate
                pred = Predicate(
                    name=rule.name,
                    patterns=rule.patterns,
                    source=rule.source,
                )
                label_predicates.append(pred)
                
                # Constraint: predicate(x) -> P(label) should be high
                # This is the implication: if rule matches, boost target label
                self.constraints.append(Constraint(
                    name=f"impl_{rule.name}_{label}",
                    constraint_type="implication",
                    predicate=pred,
                    target_label_idx=label_idx,
                    source=rule.source,
                ))
            
            self.predicates[label] = label_predicates
        
        if self.task == "action":
            self._build_action_negation_constraints()
    
    def _build_action_negation_constraints(self):
        if "Implemented" not in self.label_to_idx:
            return
        
        impl_idx = self.label_to_idx["Implemented"]
        
        # Hedging indicators (Hyland 2005) -> ¬Implemented
        hedging_pred = Predicate(
            name="hedging_indicator",
            patterns=[
                r"\b(luôn|always|hướng tới|towards|nỗ lực|strive|phấn đấu)\b",
                r"\b(tiên phong|pioneering|dẫn đầu|leading|tinh thần)\b",
                r"\b(quan tâm|care about|chú trọng|focus on|cam kết|commit)\b",
            ],
            source="Hyland (2005) Metadiscourse hedging markers",
        )
        self.constraints.append(Constraint(
            name="hedge_neg_implemented",
            constraint_type="negation",
            predicate=hedging_pred,
            target_label_idx=impl_idx,
            source="Hyland (2005): hedging -> ¬Implemented",
        ))
        
        # Vague commitment -> ¬Implemented
        vague_pred = Predicate(
            name="vague_commitment",
            patterns=[
                r"\b(nỗ lực|cố gắng|phấn đấu|hướng tới|mong muốn)\b",
                r"\b(đóng góp vào|góp phần|thúc đẩy|đẩy mạnh)\b",
            ],
            source="Vague commitment patterns -> ¬Implemented",
        )
        self.constraints.append(Constraint(
            name="vague_neg_implemented",
            constraint_type="negation",
            predicate=vague_pred,
            target_label_idx=impl_idx,
            source="Vague commitment -> ¬Implemented",
        ))
    
    def evaluate_predicates(self, text: str) -> dict[str, float]:
        scores = {label: 0.0 for label in self.labels}
        
        for label, preds in self.predicates.items():
            max_truth = 0.0
            for pred in preds:
                truth = pred.fuzzy_evaluate(text)
                max_truth = max(max_truth, truth)
            scores[label] = max_truth
        
        return scores
    
    def get_active_constraints(self, text: str) -> list[tuple[Constraint, float]]:
        active = []
        
        for constraint in self.constraints:
            if constraint.constraint_type == "exactly_one":
                active.append((constraint, 1.0))  # Always active
            elif constraint.predicate is not None:
                truth = constraint.predicate.fuzzy_evaluate(text)
                if truth > 0:
                    active.append((constraint, truth))
        
        return active
    
    def get_triggered_rule_names(self, text: str) -> dict[str, list[str]]:
        triggered = {label: [] for label in self.labels}
        
        for label, preds in self.predicates.items():
            for pred in preds:
                if pred.evaluate(text):
                    triggered[label].append(f"{pred.name} ({pred.source})")
        
        return triggered

class SemanticLoss(nn.Module):
    def __init__(
        self,
        knowledge_base: PropositionalKnowledgeBase,
        lambda_weight: float = 0.3,
        exactly_one_weight: float = 1.0,
        implication_weight: float = 0.5,
        negation_weight: float = 0.5,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.kb = knowledge_base
        self.lambda_weight = lambda_weight
        self.exactly_one_weight = exactly_one_weight
        self.implication_weight = implication_weight
        self.negation_weight = negation_weight
        self.eps = eps
    
    def exactly_one_loss(self, probs: torch.Tensor) -> torch.Tensor:
        probs = torch.clamp(probs, min=self.eps, max=1.0 - self.eps)
        one_minus_p = 1.0 - probs
        prod_all = torch.prod(one_minus_p, dim=1)
        ratio_sum = torch.sum(probs / one_minus_p, dim=1)
        wmc = prod_all * ratio_sum
        loss = -torch.log(wmc + self.eps)
        
        return loss.mean()
    
    def implication_loss(
        self,
        probs: torch.Tensor,
        texts: list[str],
    ) -> torch.Tensor:
        batch_size = probs.size(0)
        device = probs.device
        total_loss = torch.tensor(0.0, device=device)
        active_count = 0
        
        for i, text in enumerate(texts):
            for constraint in self.kb.constraints:
                if constraint.constraint_type != "implication":
                    continue
                if constraint.predicate is None or constraint.target_label_idx is None:
                    continue
                
                truth = constraint.predicate.fuzzy_evaluate(text)
                if truth <= 0:
                    continue
                
                # L_impl = -truth × log(p_k)
                target_prob = probs[i, constraint.target_label_idx]
                target_prob = torch.clamp(target_prob, min=self.eps)
                total_loss = total_loss + (-truth * torch.log(target_prob))
                active_count += 1
        
        if active_count > 0:
            return total_loss / active_count
        return total_loss
    
    def negation_loss(
        self,
        probs: torch.Tensor,
        texts: list[str],
    ) -> torch.Tensor:
        batch_size = probs.size(0)
        device = probs.device
        total_loss = torch.tensor(0.0, device=device)
        active_count = 0
        
        for i, text in enumerate(texts):
            for constraint in self.kb.constraints:
                if constraint.constraint_type != "negation":
                    continue
                if constraint.predicate is None or constraint.target_label_idx is None:
                    continue
                
                truth = constraint.predicate.fuzzy_evaluate(text)
                if truth <= 0:
                    continue
                
                target_prob = probs[i, constraint.target_label_idx]
                one_minus_p = torch.clamp(1.0 - target_prob, min=self.eps)
                total_loss = total_loss + (-truth * torch.log(one_minus_p))
                active_count += 1
        
        if active_count > 0:
            return total_loss / active_count
        return total_loss
    
    def forward(
        self,
        logits: torch.Tensor,
        texts: list[str],
    ) -> torch.Tensor:
        probs = torch.softmax(logits, dim=-1)
        loss_eo = self.exactly_one_weight * self.exactly_one_loss(probs)
        loss_impl = self.implication_weight * self.implication_loss(probs, texts)
        loss_neg = self.negation_weight * self.negation_loss(probs, texts)
        
        total = loss_eo + loss_impl + loss_neg
        
        return self.lambda_weight * total
        
@dataclass
class RuleExplanation:
    rule_name: str
    source: str
    constraint_type: str
    truth_value: float


@dataclass
class AugmentedPrediction:
    label: str
    confidence: float
    neural_label: str
    neural_confidence: float
    rule_adjusted: bool
    explanations: list[str]
    rule_scores: dict[str, float]
    active_constraints: int


class ConstrainedInference:    
    def __init__(
        self,
        knowledge_base: PropositionalKnowledgeBase,
        alpha: float = 0.3,
        eps: float = 1e-8,
    ):
        self.kb = knowledge_base
        self.alpha = alpha
        self.eps = eps
    
    def _compute_rule_log_probs(self, text: str) -> torch.Tensor:
        n_classes = len(self.kb.labels)
        scores = torch.zeros(n_classes)
        
        for constraint in self.kb.constraints:
            if constraint.predicate is None or constraint.target_label_idx is None:
                continue
            
            truth = constraint.predicate.fuzzy_evaluate(text)
            if truth <= 0:
                continue
            
            if constraint.constraint_type == "implication":
                scores[constraint.target_label_idx] += truth
            elif constraint.constraint_type == "negation":
                scores[constraint.target_label_idx] -= truth
        
        # Normalize to log-probability
        log_probs = torch.log_softmax(scores, dim=0)
        return log_probs
    
    def predict(
        self,
        logits: torch.Tensor,
        texts: list[str],
    ) -> list[AugmentedPrediction]:
        logits_cpu = logits.detach().cpu()
        
        # Neural log-probabilities
        neural_log_probs = torch.log_softmax(logits_cpu, dim=-1)
        neural_probs = torch.softmax(logits_cpu, dim=-1)
        neural_preds = torch.argmax(neural_probs, dim=-1)
        neural_confs = neural_probs.max(dim=-1).values
        
        predictions = []
        
        for i, text in enumerate(texts):
            # Rule log-probabilities
            rule_log_probs = self._compute_rule_log_probs(text)
            
            # Log-linear combination
            combined_log_probs = (
                (1 - self.alpha) * neural_log_probs[i] +
                self.alpha * rule_log_probs
            )
            
            # Normalize
            combined_probs = torch.softmax(combined_log_probs, dim=0)
            combined_pred = torch.argmax(combined_probs).item()
            combined_conf = combined_probs[combined_pred].item()
            
            neural_label = self.kb.labels[neural_preds[i].item()]
            combined_label = self.kb.labels[combined_pred]
            rule_adjusted = (neural_label != combined_label)
            
            # Build explanations
            active_constraints = self.kb.get_active_constraints(text)
            triggered_rules = self.kb.get_triggered_rule_names(text)
            
            explanations = []
            
            if rule_adjusted:
                explanations.append(
                    f"Posterior regularization: '{neural_label}' "
                    f"(neural={neural_confs[i]:.3f}) -> '{combined_label}' "
                    f"(combined={combined_conf:.3f})"
                )
            
            # List active constraints
            for constraint, truth in active_constraints:
                if constraint.constraint_type != "exactly_one" and truth > 0.3:
                    target = self.kb.labels[constraint.target_label_idx] if constraint.target_label_idx is not None else "?"
                    if constraint.constraint_type == "implication":
                        explanations.append(
                            f"α: {constraint.predicate.name} -> P({target})↑ "
                            f"[truth={truth:.2f}, src={constraint.source}]"
                        )
                    elif constraint.constraint_type == "negation":
                        explanations.append(
                            f"α: {constraint.predicate.name} -> ¬P({target}) "
                            f"[truth={truth:.2f}, src={constraint.source}]"
                        )
            
            if not explanations:
                explanations.append("No symbolic constraints active for this input.")
            
            # Rule scores for output
            rule_scores = {}
            for label in self.kb.labels:
                rule_scores[label] = round(float(torch.exp(rule_log_probs[self.kb.label_to_idx[label]])), 4)
            
            predictions.append(AugmentedPrediction(
                label=combined_label,
                confidence=combined_conf,
                neural_label=neural_label,
                neural_confidence=neural_confs[i].item(),
                rule_adjusted=rule_adjusted,
                explanations=explanations,
                rule_scores=rule_scores,
                active_constraints=len([c for c, t in active_constraints if c.constraint_type != "exactly_one"]),
            ))
        
        return predictions
    
    def predict_single(
        self, logits: torch.Tensor, text: str,
    ) -> AugmentedPrediction:
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        return self.predict(logits, [text])[0]

class SymbolicReasoner:
    TOPIC_LABELS = ["E", "S_labor", "S_community", "S_product", "G", "Non_ESG"]
    ACTION_LABELS = ["Implemented", "Planning", "Indeterminate"]
    
    def __init__(self, min_confidence: Optional[float] = None, config: Optional[dict] = None):
        topic_ns_cfg = _resolve_neuro_symbolic_config("topic", config)
        self.min_confidence = (
            min_confidence
            if min_confidence is not None
            else float(topic_ns_cfg.get("min_confidence", 0.3))
        )
        topic_labels = _resolve_task_labels("topic", config=config, labels=None)
        action_labels = _resolve_task_labels("action", config=config, labels=None)

        self.topic_kb = PropositionalKnowledgeBase(topic_labels, "topic")
        self.action_kb = PropositionalKnowledgeBase(action_labels, "action")
    
    def get_topic_constraints(self, text: str) -> dict[str, float]:
        return self.topic_kb.evaluate_predicates(text)
    
    def get_action_constraints(self, text: str) -> dict[str, float]:
        return self.action_kb.evaluate_predicates(text)
    
    def reason_topic(self, text: str, context: str = ""):
        label, confidence, matched_rules = match_topic_grounded(text, context)
        rule_scores = self.topic_kb.evaluate_predicates(text)
        triggered_rules = self.topic_kb.get_triggered_rule_names(text)
        return _SymbolicResult(
            text=text, rule_scores=rule_scores, triggered_rules=triggered_rules,
            suggested_label=label, confidence=confidence,
        )
    
    def reason_action(self, text: str, context: str = ""):
        label, confidence, matched_rules = match_actionability_grounded(text, context)
        rule_scores = self.action_kb.evaluate_predicates(text)
        triggered_rules = self.action_kb.get_triggered_rule_names(text)
        return _SymbolicResult(
            text=text, rule_scores=rule_scores, triggered_rules=triggered_rules,
            suggested_label=label, confidence=confidence,
        )
    
    def explain_prediction(
        self, text: str, pred_label: str, task: str = "topic"
    ) -> list[str]:
        kb = self.topic_kb if task == "topic" else self.action_kb
        active = kb.get_active_constraints(text)
        triggered = kb.get_triggered_rule_names(text)
        
        explanations = []
        
        if pred_label in triggered and triggered[pred_label]:
            explanations.append(
                f"Rules SUPPORT '{pred_label}': {', '.join(triggered[pred_label][:3])}"
            )
        
        for constraint, truth in active:
            if constraint.constraint_type == "exactly_one":
                continue
            if truth < 0.3:
                continue
            target = kb.labels[constraint.target_label_idx] if constraint.target_label_idx is not None else "?"
            explanations.append(
                f"  Constraint: {constraint.name} ({constraint.constraint_type}) "
                f"-> {target} [truth={truth:.2f}]"
            )
        
        if not explanations:
            explanations.append(f"No strong symbolic rules triggered for '{pred_label}'")
        
        return explanations


@dataclass
class _SymbolicResult:
    text: str
    rule_scores: dict[str, float]
    triggered_rules: dict[str, list[str]]
    suggested_label: str
    confidence: float


def _resolve_task_labels(task: str, config: Optional[dict], labels: Optional[list[str]]) -> list[str]:
    if labels:
        return labels

    if isinstance(config, dict):
        direct_labels = config.get("labels")
        if isinstance(direct_labels, list) and direct_labels:
            return direct_labels

        tasks_cfg = config.get("tasks")
        if isinstance(tasks_cfg, dict):
            task_cfg = tasks_cfg.get(task, {})
            task_labels = task_cfg.get("labels")
            if isinstance(task_labels, list) and task_labels:
                return task_labels

    if task == "topic":
        return SymbolicReasoner.TOPIC_LABELS
    return SymbolicReasoner.ACTION_LABELS


def _resolve_neuro_symbolic_config(task: str, config: Optional[dict]) -> dict:
    if not isinstance(config, dict):
        return {}

    if isinstance(config.get("neuro_symbolic"), dict):
        return config.get("neuro_symbolic", {})

    tasks_cfg = config.get("tasks")
    common_cfg = config.get("common")
    if isinstance(tasks_cfg, dict) and isinstance(common_cfg, dict):
        merged = dict(common_cfg.get("neuro_symbolic", {}) or {})
        task_cfg = tasks_cfg.get(task, {}) or {}
        task_ns = task_cfg.get("neuro_symbolic", {})
        if isinstance(task_ns, dict):
            merged.update(task_ns)
        return merged

    known_ns_keys = {
        "enabled",
        "constraint_lambda",
        "exactly_one_weight",
        "implication_weight",
        "negation_weight",
        "inference_alpha",
        "alpha",
        "min_confidence",
    }
    if any(key in config for key in known_ns_keys):
        return config

    return {}

def create_semantic_loss(
    task: str = "topic",
    lambda_weight: Optional[float] = None,
    exactly_one_weight: Optional[float] = None,
    implication_weight: Optional[float] = None,
    negation_weight: Optional[float] = None,
    labels: Optional[list[str]] = None,
    config: Optional[dict] = None,
) -> SemanticLoss:
    ns_cfg = _resolve_neuro_symbolic_config(task, config)
    resolved_labels = _resolve_task_labels(task, config=config, labels=labels)
    kb = PropositionalKnowledgeBase(resolved_labels, task)

    return SemanticLoss(
        knowledge_base=kb,
        lambda_weight=(
            float(lambda_weight)
            if lambda_weight is not None
            else float(ns_cfg.get("constraint_lambda", 0.3))
        ),
        exactly_one_weight=(
            float(exactly_one_weight)
            if exactly_one_weight is not None
            else float(ns_cfg.get("exactly_one_weight", 1.0))
        ),
        implication_weight=(
            float(implication_weight)
            if implication_weight is not None
            else float(ns_cfg.get("implication_weight", 0.5))
        ),
        negation_weight=(
            float(negation_weight)
            if negation_weight is not None
            else float(ns_cfg.get("negation_weight", 0.5))
        ),
    )


def create_constrained_inference(
    task: str = "topic",
    alpha: Optional[float] = None,
    labels: Optional[list[str]] = None,
    config: Optional[dict] = None,
) -> ConstrainedInference:
    ns_cfg = _resolve_neuro_symbolic_config(task, config)
    resolved_labels = _resolve_task_labels(task, config=config, labels=labels)
    kb = PropositionalKnowledgeBase(resolved_labels, task)
    resolved_alpha = (
        float(alpha)
        if alpha is not None
        else float(ns_cfg.get("inference_alpha", ns_cfg.get("alpha", 0.3)))
    )
    return ConstrainedInference(knowledge_base=kb, alpha=resolved_alpha)


# 1. Knowledge Base
print("\n--- Propositional Knowledge Base ---")
kb_topic = PropositionalKnowledgeBase(SymbolicReasoner.TOPIC_LABELS, "topic")
print(f"Topic KB: {len(kb_topic.constraints)} constraints")
for c in kb_topic.constraints[:5]:
    print(f"  {c.constraint_type}: {c.name}")

kb_action = PropositionalKnowledgeBase(SymbolicReasoner.ACTION_LABELS, "action")
print(f"Action KB: {len(kb_action.constraints)} constraints")

# 2. Semantic Loss
print("\n--- Semantic Loss Test ---")
sem_loss = create_semantic_loss("topic", lambda_weight=0.3)

# Simulate logits
logits = torch.randn(3, 6)
texts = [
    "Ngân hàng đã giảm phát thải CO2 được 15% so với năm 2022.",
    "Chúng tôi cam kết hướng tới phát triển bền vững.",
    "Đã triển khai chương trình đào tạo cho 5.000 nhân viên.",
]

loss = sem_loss(logits, texts)
print(f"Semantic Loss: {loss.item():.4f}")

probs = torch.softmax(logits, dim=-1)
eo_loss = sem_loss.exactly_one_loss(probs)
print(f"  Exactly-One component: {eo_loss.item():.4f}")
impl_loss = sem_loss.implication_loss(probs, texts)
print(f"  Implication component: {impl_loss.item():.4f}")
neg_loss = sem_loss.negation_loss(probs, texts)
print(f"  Negation component: {neg_loss.item():.4f}")

# 3. Constrained Inference
print("\n--- Constrained Inference Test ---")
inferencer = create_constrained_inference("topic", alpha=0.3)

preds = inferencer.predict(logits, texts)
for pred, text in zip(preds, texts):
    print(f"\n\"{text[:60]}...\"")
    print(f"  Neural: {pred.neural_label} ({pred.neural_confidence:.3f})")
    print(f"  Final:  {pred.label} ({pred.confidence:.3f})")
    print(f"  Adjusted: {pred.rule_adjusted}")
    for exp in pred.explanations[:3]:
        print(f"    {exp}")



--- Propositional Knowledge Base ---
Topic KB: 22 constraints
  exactly_one: exactly_one
  implication: impl_GRI301_Materials_E
  implication: impl_GRI302_Energy_E
  implication: impl_GRI303_Water_E
  implication: impl_GRI305_Emissions_E
Action KB: 9 constraints

--- Semantic Loss Test ---
Semantic Loss: 0.3211
  Exactly-One component: 0.7976
  Implication component: 0.5456
  Negation component: 0.0000

--- Constrained Inference Test ---

"Ngân hàng đã giảm phát thải CO2 được 15% so với năm 2022...."
  Neural: S_product (0.461)
  Final:  S_product (0.367)
  Adjusted: False
    No symbolic constraints active for this input.

"Chúng tôi cam kết hướng tới phát triển bền vững...."
  Neural: S_product (0.493)
  Final:  S_product (0.386)
  Adjusted: False
    No symbolic constraints active for this input.

"Đã triển khai chương trình đào tạo cho 5.000 nhân viên...."
  Neural: S_labor (0.451)
  Final:  S_labor (0.386)
  Adjusted: False
    α: GRI404_Training -> P(S_labor)↑ [truth=0.50, src=G

## 4. Training Logic

In [6]:
class TextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 256):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


def prepare_text(row: pd.Series, use_context_prev: bool = True, use_context_next: bool = True) -> str:
    parts = []
    if use_context_prev and row.get('ctx_prev'):
        parts.append(str(row['ctx_prev']))
    parts.append(str(row['sentence']))
    if use_context_next and row.get('ctx_next'):
        parts.append(str(row['ctx_next']))
    return ' '.join(parts)

def deep_update(base: dict, update: dict) -> dict:
    for key, value in update.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_yaml_config(config_path: Path) -> dict:
    with config_path.open('r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    if not isinstance(data, dict):
        raise ValueError(f'Invalid config format in {config_path}: root must be a mapping')
    return data


def resolve_runtime_config(
    raw_config: dict,
    task: str | None = None,
    profile: str | None = None,
    model_name: str | None = None,
    output_dir: str | None = None,
    seed: int | None = None,
) -> dict:
    defaults = raw_config.get('defaults', {})
    common = raw_config.get('common', {})
    tasks_cfg = raw_config.get('tasks', {})

    chosen_task = task or defaults.get('task')
    if chosen_task not in tasks_cfg:
        raise ValueError(
            f"Unknown task '{chosen_task}'. Available tasks: {list(tasks_cfg.keys())}"
        )

    chosen_profile = profile or defaults.get('profile', 'default')
    task_cfg = copy.deepcopy(tasks_cfg[chosen_task])
    profiles = task_cfg.pop('profiles', {})

    if chosen_profile not in profiles:
        raise ValueError(
            f"Unknown profile '{chosen_profile}' for task '{chosen_task}'. "
            f'Available profiles: {list(profiles.keys())}'
        )

    resolved = {}
    deep_update(resolved, copy.deepcopy(common))
    deep_update(resolved, task_cfg)
    deep_update(resolved, copy.deepcopy(profiles[chosen_profile]))

    if model_name:
        resolved.setdefault('model', {})['name'] = model_name
    if output_dir:
        resolved.setdefault('paths', {})['output_dir'] = output_dir

    resolved['runtime'] = {
        'task': chosen_task,
        'profile': chosen_profile,
        'seed': seed if seed is not None else defaults.get('seed', 42),
    }
    return resolved


def load_dataframe(data_path: Path) -> pd.DataFrame:
    suffix = data_path.suffix.lower()
    if suffix == '.parquet':
        return pd.read_parquet(data_path)
    if suffix == '.csv':
        return pd.read_csv(data_path)
    raise ValueError(f'Unsupported data format: {data_path}. Use .parquet or .csv')


def encode_labels(df: pd.DataFrame, label_column: str, label2id: dict[str, int]) -> pd.DataFrame:
    if label_column not in df.columns:
        raise ValueError(f"Column '{label_column}' not found. Available columns: {list(df.columns)}")
    result = df.copy()
    result["label"] = result[label_column].map(label2id)
    result = result.dropna(subset=["label"])
    result["label"] = result["label"].astype(int)
    return result


def build_text_column(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    use_context_prev = config.get("data", {}).get("use_context_prev", True)
    use_context_next = config.get("data", {}).get("use_context_next", True)
    result = df.copy()
    result["text"] = result.apply(
        lambda row: prepare_text(row, use_context_prev=use_context_prev, use_context_next=use_context_next),
        axis=1,
    )
    return result


def load_train_val_data(config: dict) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, int]]:
    train_df = build_text_column(load_dataframe(Path(config["paths"]["train_data"])), config)
    val_df = build_text_column(load_dataframe(Path(config["paths"]["val_data"])), config)

    labels = config["labels"]
    label2id = {name: idx for idx, name in enumerate(labels)}
    label_col = 'label'

    train_df = encode_labels(train_df, label_col, label2id)
    val_df = encode_labels(val_df, label_col, label2id)
    return train_df, val_df, label2id


def load_test_data(config: dict, label2id: dict[str, int]) -> pd.DataFrame | None:
    test_path = Path(config["paths"].get("test_data", ""))
    if not test_path.exists():
        return None

    test_df = build_text_column(load_dataframe(test_path), config)
    for col in config.get("test_label_candidates", ["label"]):
        if col in test_df.columns:
            return encode_labels(test_df, col, label2id)
    return None


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    micro_f1 = f1_score(labels, preds, average='micro')
    return {
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
    }


def compute_class_weights(config: dict, labels: list, method: str = 'inverse') -> torch.Tensor:
    counts = Counter(labels)
    n_samples = len(labels)
    n_classes = len(config['labels'])

    def safe_count(index: int) -> int:
        return max(counts.get(index, 0), 1)

    if method == 'inverse':
        weights = [n_samples / (n_classes * safe_count(i)) for i in range(n_classes)]
    elif method == 'sqrt_inverse':
        weights = [np.sqrt(n_samples / (n_classes * safe_count(i))) for i in range(n_classes)]
    elif method == 'effective':
        beta = 0.9999
        weights = [(1 - beta) / (1 - beta**safe_count(i)) for i in range(n_classes)]
    else:
        weights = [1.0] * n_classes

    weights = torch.tensor(weights, dtype=torch.float32)
    weights = weights / weights.sum() * n_classes
    return weights

class NeuroSymbolicTrainer(Trainer):
    def __init__(
        self,
        class_weights: torch.Tensor = None,
        semantic_loss: SemanticLoss = None,
        tokenizer_ref=None,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.semantic_loss = semantic_loss
        self.tokenizer_ref = tokenizer_ref

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fn = nn.CrossEntropyLoss()

        ce_loss = loss_fn(logits, labels)
        total_loss = ce_loss

        if self.semantic_loss is not None and self.tokenizer_ref is not None:
            texts = self.tokenizer_ref.batch_decode(inputs["input_ids"], skip_special_tokens=True)
            sem_loss = self.semantic_loss(logits, texts)
            total_loss = ce_loss + sem_loss

        return (total_loss, outputs) if return_outputs else total_loss


def build_training_args(config: dict, output_dir: Path, seed: int) -> TrainingArguments:
    train_cfg = config["training"]
    eval_batch_size = train_cfg.get("eval_batch_size", train_cfg["train_batch_size"] * 2)

    fp16_cfg = train_cfg.get("fp16", "auto")
    if isinstance(fp16_cfg, str) and fp16_cfg.lower() == "auto":
        fp16_cfg = torch.cuda.is_available()

    warmup_steps = int(train_cfg.get("warmup_steps", 0))

    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=train_cfg["epochs"],
        per_device_train_batch_size=train_cfg["train_batch_size"],
        per_device_eval_batch_size=eval_batch_size,
        learning_rate=train_cfg["learning_rate"],
        weight_decay=train_cfg.get("weight_decay"),
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=train_cfg.get("load_best_model_at_end", True),
        metric_for_best_model=train_cfg.get("metric_for_best_model", "macro_f1"),
        greater_is_better=True,
        logging_steps=50,
        gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 1),
        max_grad_norm=train_cfg.get("max_grad_norm"),
        label_smoothing_factor=train_cfg.get("label_smoothing_factor"),
        fp16=bool(fp16_cfg),
        report_to=train_cfg.get("report_to", "none"),
        seed=seed,
    )


def summarize_predictions(y_true: np.ndarray, y_pred: np.ndarray, labels: list[str]) -> dict:
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro")),
        "report": classification_report(y_true, y_pred, target_names=labels, zero_division=0),
    }


def constrained_metrics(
    logits: np.ndarray,
    texts: list[str],
    y_true: np.ndarray,
    labels: list[str],
    task: str,
    config: dict,
    alpha: float,
) -> dict:
    inferencer = create_constrained_inference(task=task, alpha=alpha, labels=labels, config=config)
    preds = inferencer.predict(torch.tensor(logits, dtype=torch.float32), texts)
    label2id = {name: idx for idx, name in enumerate(labels)}
    y_pred = np.array([label2id[p.label] for p in preds], dtype=np.int64)
    coverage = float(np.mean([1 if p.active_constraints > 0 else 0 for p in preds]))

    metrics = summarize_predictions(y_true, y_pred, labels)
    metrics["rule_coverage"] = coverage
    metrics["alpha"] = alpha
    return metrics


def train_once(config: dict) -> dict:
    task = config["runtime"]["task"]
    labels = config["labels"]
    label_col = 'label'

    seed = int(config["runtime"].get("seed", 42))
    set_seed(seed)

    print("Loading labels...")
    df_train, df_val, label2id = load_train_val_data(config)
    print(f"Train: {len(df_train)}, Val: {len(df_val)}")
    print(df_train[label_col].value_counts())

    model_name = config["model"]["name"]
    max_len = config["model"]["max_length"]
    output_dir = Path(config["paths"]["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label={i: name for i, name in enumerate(labels)},
        label2id={name: i for i, name in enumerate(labels)},
        ignore_mismatched_sizes=True,
    )

    lengths = df_train['sentence'].fillna('').astype(str).apply(lambda x: len(tokenizer.tokenize(x)))
    print(lengths.describe())

    train_dataset = TextDataset(df_train, tokenizer, max_len)
    val_dataset = TextDataset(df_val, tokenizer, max_len)

    class_weights = None
    if config["training"].get("use_class_weights", False):
        class_weights = compute_class_weights(
            config,
            df_train["label"].tolist(),
            method=config["training"].get("weight_method", "inverse"),
        )

    semantic_cfg = config.get("neuro_symbolic", {})
    semantic_loss = None
    if semantic_cfg.get("enabled", True):
        semantic_loss = create_semantic_loss(task=task, labels=labels, config=config)
        print(f"Semantic Loss enabled (lambda={semantic_cfg.get('constraint_lambda', 0.3)})")
    else:
        print("Semantic Loss disabled (baseline mode)")

    callbacks = []
    early_cfg = config.get("early_stopping", {})
    if early_cfg.get("enabled", True):
        callbacks.append(
            EarlyStoppingCallback(
                early_stopping_patience=early_cfg.get("patience", 2),
                early_stopping_threshold=early_cfg.get("threshold", 0.0),
            )
        )

    trainer = NeuroSymbolicTrainer(
        class_weights=class_weights,
        semantic_loss=semantic_loss,
        tokenizer_ref=tokenizer,
        model=model,
        args=build_training_args(config, output_dir=output_dir, seed=seed),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    print('\nTraining...')
    trainer.train()
    trainer.evaluate()

    final_path = output_dir / "final"
    trainer.save_model(str(final_path))
    tokenizer.save_pretrained(str(final_path))

    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "df_val": df_val,
        "label2id": label2id,
        "labels": labels,
    }


def evaluate_split(trainer: Trainer, dataset: TextDataset, labels: list[str]) -> tuple[np.ndarray, np.ndarray, dict]:
    preds = trainer.predict(dataset)
    logits = preds.predictions
    y_pred = np.argmax(logits, axis=-1)
    y_true = preds.label_ids
    return logits, y_true, summarize_predictions(y_true, y_pred, labels)

def run_pipeline(config: dict, alpha_grid: list[float] | None = None) -> dict:
    run = train_once(config)
    trainer = run["trainer"]
    tokenizer = run["tokenizer"]
    labels = run["labels"]
    task = config["runtime"]["task"]

    max_len = config["model"]["max_length"]
    val_dataset = TextDataset(run["df_val"], tokenizer, max_len)
    val_logits, val_y_true, val_neural = evaluate_split(trainer, val_dataset, labels)

    result = {
        "mode": "neuro" if config.get("neuro_symbolic", {}).get("enabled", True) else "baseline",
        "task": task,
        "val_neural": val_neural,
    }

    test_df = load_test_data(config, run["label2id"])
    if test_df is not None and len(test_df) > 0:
        test_dataset = TextDataset(test_df, tokenizer, max_len)
        test_logits, test_y_true, test_neural = evaluate_split(trainer, test_dataset, labels)
        result["test_neural"] = test_neural
    else:
        test_df, test_logits, test_y_true = None, None, None

    use_neuro = config.get("neuro_symbolic", {}).get("enabled", True)
    if use_neuro:
        grid = alpha_grid or config.get("experiment", {}).get("alpha_grid", [0.3, 0.5, 0.7, 0.9])
        print(f"Tuning alpha on validation set with grid: {grid}")
        alpha_search = []
        for alpha in grid:
            alpha_search.append(
                constrained_metrics(
                    logits=val_logits,
                    texts=run["df_val"]["text"].tolist(),
                    y_true=val_y_true,
                    labels=labels,
                    task=task,
                    config=config,
                    alpha=float(alpha),
                )
            )

        best = max(alpha_search, key=lambda x: x["macro_f1"]) if alpha_search else None
        result["val_constrained_search"] = alpha_search
        result["best_alpha"] = best["alpha"] if best else None

        if test_df is not None and len(test_df) > 0 and best is not None:
            result["test_constrained"] = constrained_metrics(
                logits=test_logits,
                texts=test_df["text"].tolist(),
                y_true=test_y_true,
                labels=labels,
                task=task,
                config=config,
                alpha=float(best["alpha"]),
            )

    output_dir = Path(config["paths"]["output_dir"])
    summary_path = output_dir / "metrics_summary.json"
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Saved metrics summary to: {summary_path}")

    return result


def run_compare_experiment(config: dict, alpha_grid: list[float] | None = None) -> dict:
    root_output = Path(config["paths"]["output_dir"])

    baseline_cfg = copy.deepcopy(config)
    baseline_cfg.setdefault("neuro_symbolic", {})["enabled"] = False
    baseline_cfg["paths"]["output_dir"] = str(root_output / "baseline")

    neuro_cfg = copy.deepcopy(config)
    neuro_cfg.setdefault("neuro_symbolic", {})["enabled"] = True
    neuro_cfg["paths"]["output_dir"] = str(root_output / "neuro")

    print("\n=== Running baseline (no neuro-symbolic) ===")
    baseline_result = run_pipeline(baseline_cfg, alpha_grid=alpha_grid)

    print("\n=== Running neuro-symbolic ===")
    neuro_result = run_pipeline(neuro_cfg, alpha_grid=alpha_grid)

    comparison = {
        "task": config["runtime"]["task"],
        "baseline": baseline_result,
        "neuro_symbolic": neuro_result,
    }

    compare_path = root_output / "comparison_summary.json"
    compare_path.parent.mkdir(parents=True, exist_ok=True)
    with compare_path.open("w", encoding="utf-8") as f:
        json.dump(comparison, f, ensure_ascii=False, indent=2)
    print(f"Saved comparison summary to: {compare_path}")

    return comparison


def parse_alpha_grid(alpha_grid_raw: str | None) -> list[float] | None:
    if not alpha_grid_raw:
        return None
    return [float(x.strip()) for x in alpha_grid_raw.split(",") if x.strip()]

def train_for_notebook(task, profile, output_dir, run_experiment):
    run_config=resolve_runtime_config(
        raw_config=config,
        task=task,
        profile=profile,
        output_dir=output_dir,
    )

    alpha_grid = parse_alpha_grid(alpha_grid_raw=None)

    if run_experiment:
        run_compare_experiment(run_config, alpha_grid=alpha_grid)
    else:
        run_pipeline(run_config, alpha_grid=alpha_grid)

## 5. Run Training

In [7]:
# 1. Train Topic Classification
train_for_notebook(task='topic',run_experiment=True, profile=None, output_dir=None)


=== Running baseline (no neuro-symbolic) ===
Loading labels...


Train: 91188, Val: 11399
label
5    65338
4    13450
1     4470
3     3904
0     2429
2     1597
Name: count, dtype: int64


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tuning alpha on validation set with grid: [0.3, 0.5, 0.7, 0.9]
Saved metrics summary to: /kaggle/working/output/models/topic_classifier/neuro/metrics_summary.json
Saved comparison summary to: /kaggle/working/output/models/topic_classifier/comparison_summary.json


In [9]:
# 2. Train Actionability Classification
train_for_notebook(task='action',run_experiment=True, profile=None, output_dir=None)


=== Running baseline (no neuro-symbolic) ===
Loading labels...
Train: 25808, Val: 3226
label
2    16756
0     8322
1      730
Name: count, dtype: int64


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

count    25808.000000
mean        48.218963
std        204.668236
min          2.000000
25%         20.000000
50%         36.000000
75%         57.000000
max      31607.000000
Name: sentence, dtype: float64
Semantic Loss disabled (baseline mode)

Training...


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1
1,0.531650,0.527134,0.660718,0.782300
2,0.437036,0.509162,0.695363,0.812908
3,0.345409,0.550265,0.701787,0.813757
4,0.294554,0.616576,0.699019,0.809957
5,0.244115,0.687495,0.714741,0.827477


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tuning alpha on validation set with grid: [0.3, 0.5, 0.7, 0.9]
Saved metrics summary to: /kaggle/working/output/models/action_classifier/neuro/metrics_summary.json
Saved comparison summary to: /kaggle/working/output/models/action_classifier/comparison_summary.json
